# Домашнее задание № 2. Мешок слов

## Задание 1 (3 балла)

У векторайзеров в sklearn есть встроенная токенизация на регулярных выражениях. Найдите способо заменить её на кастомную токенизацию

Обучите векторайзер с дефолтной токенизацией, с токенизацией razdel.tokenize и с токенизацией+лемматизацией из mystem. Обучите классификатор (любой) с каждым из векторизаторов. Сравните метрики и выберите победителя. 

(в вашей тетрадке должен быть код обучения и все метрики; если вы сдаете в .py файлах то сохраните полученные метрики в отдельном файле или в комментариях)

In [2]:
%pip install -q pandas scikit-learn razdel pymystem3 datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 111
pd.set_option('display.max_colwidth', None)

In [ ]:
def get_metrics(y_true, preds):
    """Метрики по токсичному классу (1), а также accuracy и macro F1"""
    return {
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'f1': f1_score(y_true, preds, zero_division=0),
        'macro_f1': f1_score(y_true, preds, average='macro', zero_division=0),
        'accuracy': accuracy_score(y_true, preds),
    }

In [6]:
data = pd.read_csv('labeled.csv')
data['toxic'] = data['toxic'].astype(int)

print(data.shape)
print(data.toxic.value_counts(normalize=True))
data.head()

(14412, 2)
toxic
0    0.66514
1    0.33486
Name: proportion, dtype: float64


,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1
1,"Хохлы, это отдушина затюканого россиянина, мол, вон, а у хохлов еще хуже. Если бы хохлов не было, кисель их бы придумал.\n",1
2,Собаке - собачья смерть\n,1
3,"Страницу обнови, дебил. Это тоже не оскорбление, а доказанный факт - не-дебил про себя во множественном числе писать не будет. Или мы в тебя верим - это ты и твои воображаемые друзья?\n",1
4,"тебя не убедил 6-страничный пдф в том, что Скрипалей отравила Россия? Анализировать и думать пытаешься? Ватник что ли?)\n",1


In [7]:
train, test = train_test_split(
    data, test_size=0.1, shuffle=True, stratify=data.toxic, random_state=RANDOM_STATE
)
y_train, y_test = train.toxic.values, test.toxic.values
train.shape, test.shape

((12970, 2), (1442, 2))

In [8]:
from razdel import tokenize
def razdel_tokenizer(text):
    return [token.text for token in tokenize(text) if re.search(r'\w', token.text)]

razdel_tokenizer('Кто-то писал это в 2019-м году, т.е. давно!')

['Кто-то', 'писал', 'это', 'в', '2019-м', 'году', 'т', 'е', 'давно']

In [ ]:
from pymystem3 import Mystem

mystem = Mystem()

def mystem_tokenizer(text):
    """Токенизация и лемматизация с помощью mystem"""
    return [lemma.strip() for lemma in mystem.lemmatize(text) if re.search(r'\w', lemma)]

mystem_tokenizer('Кошки сидели на окнах и смотрели на прохожих')

['кошка', 'сидеть', 'на', 'окно', 'и', 'смотреть', 'на', 'прохожий']

In [13]:
import json
import subprocess

from pymystem3.constants import MYSTEM_BIN

SEP = 'qqsepqq'


def lemmatize_corpus(texts):
    """Лемматизирует все тексты за один запуск mystem"""
    texts = [' '.join(text.lower().split()) for text in texts]
    payload = ''.join(f'{text}\n{SEP}\n' for text in texts)

    output = subprocess.run(
        [MYSTEM_BIN, '--format', 'json', '-c', '-d'],
        input=payload.encode('utf-8'), capture_output=True, check=True,
    ).stdout.decode('utf-8')

    decoder = json.JSONDecoder()
    docs, current, pos = [], [], 0
    while pos < len(output):
        if output[pos].isspace():
            pos += 1
            continue
        tokens, pos = decoder.raw_decode(output, pos)
        for token in tokens:
            if token['text'].strip() == SEP:
                docs.append(' '.join(current))
                current = []
                continue
            analysis = token.get('analysis')
            lemma = (analysis[0]['lex'] if analysis else token['text']).strip()
            if re.search(r'\w', lemma):
                current.append(lemma)

    assert len(docs) == len(texts), f'Получено {len(docs)} текстов вместо {len(texts)}'
    return docs

train_lemmas = lemmatize_corpus(train.comment)
test_lemmas = lemmatize_corpus(test.comment)

In [14]:
variants = {
    'default': (train.comment, test.comment, {}),
    'razdel': (train.comment, test.comment, {'tokenizer': razdel_tokenizer, 'token_pattern': None}),
    'mystem (лемматизация)': (train_lemmas, test_lemmas, {'tokenizer': str.split, 'token_pattern': None}),
}

results_1 = {}
for name, (texts_train, texts_test, tokenizer_params) in variants.items():
    vectorizer = TfidfVectorizer(min_df=2, max_df=0.5, **tokenizer_params)
    X_train = vectorizer.fit_transform(texts_train)
    X_test = vectorizer.transform(texts_test)

    clf = LogisticRegression(C=5.0, class_weight='balanced', max_iter=1000)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    results_1[name] = {**get_metrics(y_test, preds), 'vocab_size': len(vectorizer.vocabulary_)}
    print(f'{name}')
    print(classification_report(y_test, preds, digits=3, zero_division=0))

default
              precision    recall  f1-score   support

           0      0.898     0.904     0.901       959
           1      0.807     0.797     0.802       483

    accuracy                          0.868      1442
   macro avg      0.853     0.851     0.852      1442
weighted avg      0.868     0.868     0.868      1442

razdel
              precision    recall  f1-score   support

           0      0.906     0.901     0.903       959
           1      0.805     0.814     0.809       483

    accuracy                          0.872      1442
   macro avg      0.855     0.857     0.856      1442
weighted avg      0.872     0.872     0.872      1442

mystem (лемматизация)
              precision    recall  f1-score   support

           0      0.910     0.918     0.914       959
           1      0.834     0.820     0.827       483

    accuracy                          0.885      1442
   macro avg      0.872     0.869     0.870      1442
weighted avg      0.884     0.885    

In [15]:
table_1 = pd.DataFrame(results_1).T.sort_values('f1', ascending=False)
table_1['vocab_size'] = table_1['vocab_size'].astype(int)
table_1.to_csv('metrics_task1.csv')
display(table_1.round(3))

winner = table_1.index[0]
gap = table_1.f1.iloc[0] - table_1.f1.iloc[1]
print(f'Победитель по F1: {winner} (F1 = {table_1.f1.iloc[0]:.3f}, отрыв от второго места {gap:.3f})')

,precision,recall,f1,macro_f1,accuracy,vocab_size
mystem (лемматизация),0.834,0.820,0.827,0.870,0.885,14626
razdel,0.805,0.814,0.809,0.856,0.872,23769
default,0.807,0.797,0.802,0.852,0.868,23745


Победитель по F1: mystem (лемматизация) (F1 = 0.827, отрыв от второго места 0.017)


## Задание 2 (3 балла)

Обучите 2 любых разных классификатора (используя алгоритмы из семинара). Используйте eng сабсет из `mteb/multi-hatecheck` в качестве датасета. 

Предскажите токсичность (`is_hateful`) для текстов из тестовой выборки (используйте одну и ту же выборку для обоих классификаторов) и найдите 10 самых токсичных для каждого из классификаторов (по вероятности). Сравните получаемые тексты - какие тексты совпадают, какие отличаются, правда ли тексты токсичные?

Требования к моделям:   
а) один классификатор должен использовать CountVectorizer, другой TfidfVectorizer  
б) у векторазера должны быть вручную заданы как минимум 5 параметров (можно ставить разные параметры tfidfvectorizer и countvectorizer)  
в) у классификатора должно быть задано вручную как минимум 2 параметра (по возможности)  
г)  f1 мера каждого из классификаторов должна быть минимум 0.60 

*random_seed, encoding, decode_error не считаются за параметры

In [16]:
from datasets import load_dataset

hatecheck = load_dataset('mteb/multi-hatecheck')
eng = hatecheck['test'].filter(lambda e: e['lang'] == 'eng').to_pandas()
eng = eng.drop_duplicates(subset='text').reset_index(drop=True)

assert set(eng.is_hateful.unique()) == {'hateful', 'non-hateful'}, eng.is_hateful.unique()
eng['label'] = (eng.is_hateful == 'hateful').astype(int)

print(eng.shape)
print(eng.label.value_counts(normalize=True))
eng.head()

c:\Users\1\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(3728, 5)
label
1    0.6875
0    0.3125
Name: proportion, dtype: float64


,functionality,text,is_hateful,lang,label
0,derog_neg_emote_h,I hate women.,hateful,eng,1
1,derog_neg_emote_h,I hate trans people.,hateful,eng,1
2,derog_neg_emote_h,I hate gay people.,hateful,eng,1
3,derog_neg_emote_h,I hate black people.,hateful,eng,1
4,derog_neg_emote_h,I hate disabled people.,hateful,eng,1


In [17]:
hc_train, hc_test = train_test_split(
    eng, test_size=0.2, shuffle=True, stratify=eng.label, random_state=RANDOM_STATE
)
hc_y_train, hc_y_test = hc_train.label.values, hc_test.label.values
hc_train.shape, hc_test.shape

((2982, 5), (746, 5))

In [18]:
dummy = DummyClassifier(strategy='constant', constant=1).fit(hc_train.text, hc_y_train)
pd.Series(get_metrics(hc_y_test, dummy.predict(hc_test.text))).round(3)

precision    0.688
recall       1.000
f1           0.815
macro_f1     0.407
accuracy     0.688
dtype: float64

In [19]:
count_vec = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    max_features=10000,
    binary=True,
)
X_train_count = count_vec.fit_transform(hc_train.text)
X_test_count = count_vec.transform(hc_test.text)

nb = MultinomialNB(alpha=0.5, fit_prior=False)
nb.fit(X_train_count, hc_y_train)

nb_preds = nb.predict(X_test_count)
print(classification_report(hc_y_test, nb_preds, digits=3, zero_division=0))

              precision    recall  f1-score   support

           0      0.870     0.833     0.851       233
           1      0.925     0.943     0.934       513

    accuracy                          0.909       746
   macro avg      0.898     0.888     0.893       746
weighted avg      0.908     0.909     0.908       746



In [20]:
tfidf_vec = TfidfVectorizer(
    lowercase=True,
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.8,
    max_features=20000,
    sublinear_tf=True,
)
X_train_tfidf = tfidf_vec.fit_transform(hc_train.text)
X_test_tfidf = tfidf_vec.transform(hc_test.text)

logreg = LogisticRegression(C=3.0, class_weight='balanced', max_iter=1000)
logreg.fit(X_train_tfidf, hc_y_train)

logreg_preds = logreg.predict(X_test_tfidf)
print(classification_report(hc_y_test, logreg_preds, digits=3, zero_division=0))

              precision    recall  f1-score   support

           0      0.974     0.961     0.968       233
           1      0.983     0.988     0.985       513

    accuracy                          0.980       746
   macro avg      0.978     0.975     0.977       746
weighted avg      0.980     0.980     0.980       746



In [21]:
table_2 = pd.DataFrame({
    'CountVectorizer + NB': get_metrics(hc_y_test, nb_preds),
    'TfidfVectorizer + LogReg': get_metrics(hc_y_test, logreg_preds),
}).T
table_2.to_csv('metrics_task2.csv')
display(table_2.round(3))
assert (table_2.f1 >= 0.60).all(), 'F1 одного из классификаторов ниже 0.60'

,precision,recall,f1,macro_f1,accuracy
CountVectorizer + NB,0.925,0.943,0.934,0.893,0.909
TfidfVectorizer + LogReg,0.983,0.988,0.985,0.977,0.980


In [22]:
def top_toxic(probas, n=10):
    result = hc_test[['text', 'functionality', 'label']].copy()
    result['proba'] = probas
    return result.sort_values('proba', ascending=False, kind='stable').head(n)


top_nb = top_toxic(nb.predict_proba(X_test_count)[:, 1])
top_lr = top_toxic(logreg.predict_proba(X_test_tfidf)[:, 1])

print('CountVectorizer + Naive Bayes')
display(top_nb)

print('TfidfVectorizer + Logistic Regression')
display(top_lr)

CountVectorizer + Naive Bayes


,text,functionality,label,proba
1596,We're talking about gay people here. They can't help themselves but be complete dickheads.,ref_subs_sent_h,1,1.0
1594,We're talking about women here. They can't help themselves but be complete dickheads.,ref_subs_sent_h,1,1.0
1518,Trans people can be kind to you at times. Don't forget they're still just worthless scum.,ref_subs_sent_h,1,1.0
1884,Is it easy to tell that I am disgusted by all the black people around here?,phrase_question_h,1,1.0
1523,Immigrants can be kind to you at times. Don't forget they're still just worthless scum.,ref_subs_sent_h,1,1.0
495,It baffles me why people care even the slightest about the lives of Muslims.,derog_impl_h,1,1.0
1498,A gay person must have committed that crime. Only they can be that violent and cruel.,ref_subs_sent_h,1,1.0
2011,When will it finally be clear to people that it's in a disabled person's nature to be an annoying prick?,phrase_question_h,1,1.0
1984,Isn't it apparent that we would be better off if those Muslims were eradicated?,phrase_question_h,1,1.0
1559,We should simply shoot them. They're just women after all.,ref_subs_sent_h,1,1.0


TfidfVectorizer + Logistic Regression


,text,functionality,label,proba
3406,All black people like them are v i l e.,spell_space_add_h,1,0.990782
3407,All disabled people like them are v i l e.,spell_space_add_h,1,0.988605
314,Immigrants are nothing but refuse.,derog_dehum_h,1,0.987497
3502,Typical s h e m a l e behaviour.,spell_space_add_h,1,0.987338
3434,Black people are a c a n c e r.,spell_space_add_h,1,0.987190
1559,We should simply shoot them. They're just women after all.,ref_subs_sent_h,1,0.980614
3514,Typical w e t b a c k behaviour.,spell_space_add_h,1,0.979993
3384,I h a t e gay people.,spell_space_add_h,1,0.979952
3506,Typical n i g g e r behaviour.,spell_space_add_h,1,0.976938
3214,Trans people can all go fuk themselves.,spell_char_del_h,1,0.976182


In [23]:
common = top_nb.index.intersection(top_lr.index)
only_nb = top_nb.index.difference(top_lr.index)
only_lr = top_lr.index.difference(top_nb.index)

for title, idx in [('Совпадают', common), ('Только у NB', only_nb), ('Только у LogReg', only_lr)]:
    print(f'\n{title}: {len(idx)}')
    for text in eng.loc[idx, 'text']:
        print('  -', text)

print(f'\nДействительно токсичные по разметке: NB {top_nb.label.sum()}/10, LogReg {top_lr.label.sum()}/10')


Совпадают: 1
  - We should simply shoot them. They're just women after all. 

Только у NB: 9
  - It baffles me why people care even the slightest about the lives of Muslims. 
  - A gay person must have committed that crime. Only they can be that violent and cruel. 
  - Trans people can be kind to you at times. Don't forget they're still just worthless scum. 
  - Immigrants can be kind to you at times. Don't forget they're still just worthless scum. 
  - We're talking about women here. They can't help themselves but be complete dickheads. 
  - We're talking about gay people here. They can't help themselves but be complete dickheads. 
  - Is it easy to tell that I am disgusted by all the black people around here?
  - Isn't it apparent that we would be better off if those Muslims were eradicated?
  - When will it finally be clear to people that it's in a disabled person's nature to be an annoying prick?

Только у LogReg: 9
  - Immigrants are nothing but refuse. 
  - Trans people can all 

## Задание 3 (2 балла - 0,5 балл за каждый классификатор)

Для классификаторов Logistic Regression, Decision Trees, Naive Bayes, RandomForest найдите способ извлечь важность признаков для предсказания токсичного класса (любой из датасетов). Сопоставьте полученные числа со словами (или нграммами) в словаре и найдите топ - 5 "токсичных" слов для каждого из классификаторов. 

Важное требование: в топе не должно быть стоп-слов. Для этого вам нужно будет правильным образом настроить векторизацию. 
Также как и в предыдущем задании у классификаторов должно быть задано вручную как минимум 2 параметра (по возможности, f1 мера каждого из классификаторов должна быть минимум 0.6)

In [ ]:
extra_stop_words = {
    'ain', 'aren', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn',
    'mustn', 'needn', 'shan', 'shouldn', 'wasn', 'weren', 'won', 'wouldn', 'll', 've', 're',
    'im', 'ive', 'dont', 'cant', 'just', 'really', 'did', 'does', 'doing',
}
stop_words = sorted(ENGLISH_STOP_WORDS.union(extra_stop_words))

vectorizer_3 = CountVectorizer(
    lowercase=True,
    stop_words=stop_words,
    token_pattern=r'(?u)\b[a-z]{2,}\b',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
)
X_train_3 = vectorizer_3.fit_transform(hc_train.text)
X_test_3 = vectorizer_3.transform(hc_test.text)
feature_names = vectorizer_3.get_feature_names_out()

assert not any(word in stop_words for feature in feature_names for word in feature.split())
X_train_3.shape

(2982, 947)

In [25]:
presence = X_train_3 > 0
share_toxic = np.asarray(presence[hc_y_train == 1].mean(axis=0)).ravel()
share_clean = np.asarray(presence[hc_y_train == 0].mean(axis=0)).ravel()
toxic_mask = share_toxic > share_clean


def top_features(scores, mask=None, n=5):
    table = pd.DataFrame({
        'feature': feature_names,
        'importance': scores,
        'share_toxic': share_toxic,
        'share_clean': share_clean,
    })
    if mask is not None:
        table = table[mask]
    table = table[table.importance > 0]
    return table.sort_values('importance', ascending=False).head(n).reset_index(drop=True)


tops, results_3 = {}, {}


def fit_and_report(name, clf):
    clf.fit(X_train_3, hc_y_train)
    preds = clf.predict(X_test_3)
    results_3[name] = get_metrics(hc_y_test, preds)
    print(classification_report(hc_y_test, preds, digits=3, zero_division=0))
    return clf

In [26]:
lr_3 = fit_and_report('Logistic Regression',
                      LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000))
tops['Logistic Regression'] = top_features(lr_3.coef_[0])
tops['Logistic Regression']

tree_3 = fit_and_report('Decision Tree',
                        DecisionTreeClassifier(max_depth=40, min_samples_leaf=2, random_state=RANDOM_STATE))
tops['Decision Tree'] = top_features(tree_3.feature_importances_, mask=toxic_mask)
tops['Decision Tree']

nb_3 = fit_and_report('Naive Bayes', MultinomialNB(alpha=1.0, fit_prior=True))
log_ratio = nb_3.feature_log_prob_[1] - nb_3.feature_log_prob_[0]
tops['Naive Bayes'] = top_features(log_ratio)
tops['Naive Bayes']

rf_3 = fit_and_report('Random Forest',
                      RandomForestClassifier(n_estimators=300, max_depth=60, min_samples_leaf=2,
                                             random_state=RANDOM_STATE, n_jobs=-1))
tops['Random Forest'] = top_features(rf_3.feature_importances_, mask=toxic_mask)
tops['Random Forest']

              precision    recall  f1-score   support

           0      0.867     0.867     0.867       233
           1      0.940     0.940     0.940       513

    accuracy                          0.917       746
   macro avg      0.903     0.903     0.903       746
weighted avg      0.917     0.917     0.917       746

              precision    recall  f1-score   support

           0      0.711     0.687     0.699       233
           1      0.860     0.873     0.867       513

    accuracy                          0.815       746
   macro avg      0.785     0.780     0.783       746
weighted avg      0.813     0.815     0.814       746

              precision    recall  f1-score   support

           0      0.881     0.601     0.714       233
           1      0.842     0.963     0.898       513

    accuracy                          0.850       746
   macro avg      0.861     0.782     0.806       746
weighted avg      0.854     0.850     0.841       746

              preci

,feature,importance,share_toxic,share_clean
0,people,0.024607,0.401951,0.304721
1,typical,0.014197,0.046829,0.017167
2,country,0.010961,0.052683,0.023605
3,immigrants,0.010441,0.094146,0.053648
4,women,0.008403,0.091220,0.061159


In [27]:
table_3 = pd.DataFrame(results_3).T
table_3.to_csv('metrics_task3.csv')
display(table_3.round(3))
assert (table_3.f1 >= 0.60).all(), 'F1 одного из классификаторов ниже 0.60'

summary = pd.DataFrame({name: top.feature for name, top in tops.items()})
summary.index = range(1, len(summary) + 1)
summary

,precision,recall,f1,macro_f1,accuracy
Logistic Regression,0.940,0.940,0.940,0.903,0.917
Decision Tree,0.860,0.873,0.867,0.783,0.815
Naive Bayes,0.842,0.963,0.898,0.806,0.850
Random Forest,0.796,0.990,0.883,0.743,0.819


,Logistic Regression,Decision Tree,Naive Bayes,Random Forest
1,country,muslims,typical behaviour,people
2,immigrants,people,head,typical
3,muslims,typical,fucked head,country
4,women,women,common,immigrants
5,typical,help,like deserve,women
